# 16 — Ingestão da base real (`montar_base`)

Desenvolve o passo que baixa Ibovespa+CDI e popula o SQLite. **F1, NF6.**

In [1]:
import sys, os, tempfile
_cwd = os.getcwd()
RAIZ = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'tests' else _cwd 
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ) 
import os
from app import dal

## Desenvolvimento

A(s) função(ões)/classe(s) abaixo foi(ram) escrita(s) aqui e, após os testes, movida(s) para `app/ingestao.py`.

In [2]:
BANCO_PADRAO = {"1mo": "data/mercado.db", "1d": "data/mercado_diario.db"}


def montar_base(db_path: str | None = None,
                inicio: str = "2000-01-01",
                fim: str | None = None,
                frequencia: str = "1mo") -> dict:
    """Baixa Ibovespa (Yahoo) + CDI (Banco Central) e grava o SQLite.

    Parameters
    ----------
    db_path : caminho do banco (``None`` => o padrão da frequência).
    frequencia : ``"1mo"`` (mensal) ou ``"1d"`` (diário).

    Returns
    -------
    dict com ``db_path``, ``frequencia``, ``n_periodos`` e ``periodo``
    (primeira e última data).
    """
    if db_path is None:
        db_path = BANCO_PADRAO[frequencia]
    os.makedirs(os.path.dirname(db_path) or ".", exist_ok=True)

    # 1. Ibovespa — níveis (tabela 'ibovespa').
    precos = dal.baixar_precos(["^BVSP"], inicio, fim, frequencia=frequencia)
    precos = precos.rename(columns={precos.columns[1]: "fechamento"})
    dal.gravar_sqlite(precos, db_path, "ibovespa")

    # 2. CDI — taxa por período (tabela 'cdi').
    cdi = dal.baixar_cdi_bcb(inicio, fim, frequencia=frequencia)
    dal.gravar_sqlite(cdi, db_path, "cdi")

    # 3. Retornos alinhados por data (tabela 'retornos'). O merge interno
    #    descarta feriado de bolsa que não é feriado bancário, e vice-versa.
    ret_ibov = dal.calcular_retornos(precos.rename(columns={"fechamento": "ibov"}))
    retornos = ret_ibov.merge(cdi, on="data", how="inner")
    if retornos.empty:
        raise ValueError("Sem datas em comum entre Ibovespa e CDI — verifique o período.")
    dal.gravar_sqlite(retornos, db_path, "retornos")

    return {"db_path": db_path,
            "frequencia": frequencia,
            "n_periodos": len(retornos),
            "periodo": (retornos["data"].iloc[0], retornos["data"].iloc[-1])}

**Teste** — monta o banco (download real, protegido).

In [3]:
import tempfile
try:
    dbm = os.path.join(tempfile.gettempdir(), 'dev16.db')
    info = montar_base(dbm, '2020-01-01', '2020-06-01')
    print('montar_base ->', info); print(dal.ler_sqlite(dbm,'retornos'))
    assert set(dal.ler_sqlite(dbm,'retornos').columns) == {'data','ibov','cdi'}
    os.remove(dbm); print('ingestao: PASSOU')
except Exception as e:
    print('offline:', type(e).__name__, e)

montar_base -> {'db_path': 'C:\\Users\\MURILO\\AppData\\Local\\Temp\\dev16.db', 'frequencia': '1mo', 'n_periodos': 4, 'periodo': ('2020-02', '2020-05')}
      data      ibov     cdi
0  2020-02 -0.084291  0.0029
1  2020-03 -0.299044  0.0034
2  2020-04  0.102520  0.0028
3  2020-05  0.085671  0.0024
ingestao: PASSOU


In [4]:
# --- ingestao DIARIA ---
from app import ingestao as _ing
assert _ing.BANCO_PADRAO == {'1mo': 'data/mercado.db', '1d': 'data/mercado_diario.db'}

# (a) banco diario versionado no repo: valida a estrutura sem tocar na rede
dbd = os.path.join(RAIZ, 'data', 'mercado_diario.db')
if os.path.exists(dbd):
    rd = dal.ler_sqlite(dbd, 'retornos')
    print(f'base diaria: {len(rd)} pregoes, {rd["data"].iloc[0]} a {rd["data"].iloc[-1]}')
    assert set(rd.columns) == {'data', 'ibov', 'cdi'}
    assert rd['data'].str.match(r'^\d{4}-\d{2}-\d{2}$').all()       # AAAA-MM-DD
    assert rd['data'].is_unique and rd['data'].is_monotonic_increasing
    assert not rd.isna().any().any() and len(rd) > 500
    print('ingestao diaria (banco versionado): PASSOU')
else:
    print('data/mercado_diario.db ausente -> python -m app.ingestao 2022-05-22 --diario')


base diaria: 1050 pregoes, 2022-05-24 a 2026-08-05
ingestao diaria (banco versionado): PASSOU


In [5]:
# (b) download diario de verdade, se houver rede
try:
    dbd_tmp = os.path.join(tempfile.gettempdir(), 'dev16d.db')
    info_d = _ing.montar_base(dbd_tmp, '2024-01-01', '2024-03-01', frequencia='1d')
    print('montar_base(1d) ->', info_d)
    assert info_d['frequencia'] == '1d' and info_d['n_periodos'] > 20
    assert len(info_d['periodo'][0]) == 10                            # AAAA-MM-DD
    os.remove(dbd_tmp); print('montar_base(frequencia="1d") online: PASSOU')
except Exception as e:
    print('offline:', type(e).__name__, e)

montar_base(1d) -> {'db_path': 'C:\\Users\\MURILO\\AppData\\Local\\Temp\\dev16d.db', 'frequencia': '1d', 'n_periodos': 40, 'periodo': ('2024-01-03', '2024-02-29')}
montar_base(frequencia="1d") online: PASSOU
